# Nova AI — Free GPU Image Generation (Google Colab)

Run this notebook on **Google Colab with GPU** (Runtime → Change runtime type → T4 GPU).

Your PC stays cool — Colab's GPU generates images. Nova V2 on your PC calls this via a temporary URL.

**Steps:**
1. Upload/open this notebook in [Google Colab](https://colab.research.google.com)
2. Runtime → Change runtime type → **GPU**
3. Run all cells
4. Copy the **public URL** from the last cell
5. In `AImodel/v2_llm_agent/.env` set:
   ```
   IMAGE_PROVIDER=openai,sd-webui
   SD_WEBUI_URL=https://xxxx.ngrok-free.app
   ```
6. Restart V2 (`restart-v2.bat`)

Note: Colab free sessions disconnect after ~90 min idle. Re-run the notebook to get a new URL.
For daily use, **OpenAI DALL-E** (already in your `.env`) is simpler — no Colab needed.

In [ ]:
# Check GPU
!nvidia-smi || echo 'WARNING: No GPU — enable Runtime → Change runtime type → GPU'

In [ ]:
# Install Stable Diffusion WebUI (first run ~5-10 min)
!git clone -q https://github.com/AUTOMATIC1111/stable-diffusion-webui.git /content/sd-webui
%cd /content/sd-webui
!pip install -q ngrok pyngrok

In [ ]:
# Download SD 1.5 model (~4GB, one time per session)
!mkdir -p models/Stable-diffusion
!wget -q -O models/Stable-diffusion/v1-5-pruned-emaonly.safetensors \
  https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors \
  || echo 'If wget fails, download manually from Hugging Face'

In [ ]:
# Start WebUI API in background (GPU — fast, ~5-15 sec per image)
import subprocess, time, os
os.chdir('/content/sd-webui')
env = os.environ.copy()
env['COMMANDLINE_ARGS'] = '--api --listen --port 7860 --xformers --no-half-vae'
proc = subprocess.Popen(['python', 'launch.py'], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print('Starting WebUI (wait ~2-3 min for model load)...')
time.sleep(120)
print('WebUI should be ready on port 7860')

In [ ]:
# Expose to internet via ngrok (paste URL into V2 .env as SD_WEBUI_URL)
from pyngrok import ngrok

# Optional: free ngrok token from https://dashboard.ngrok.com/get-started/your-authtoken
# ngrok.set_auth_token('your_token_here')

tunnel = ngrok.connect(7860, "http")
public_url = str(tunnel).split('"')[1] if '"' in str(tunnel) else str(tunnel)
print('=' * 60)
print('NOVA V2 CONFIG — add to v2_llm_agent/.env:')
print(f'SD_WEBUI_URL={public_url}')
print('IMAGE_PROVIDER=openai,sd-webui')
print('=' * 60)
print('Keep this Colab tab open while generating images.')

In [ ]:
# Quick test — generate one image on Colab GPU
import requests, base64
from IPython.display import Image, display

r = requests.post('http://127.0.0.1:7860/sdapi/v1/txt2img', json={
    'prompt': 'sports car, photorealistic, studio lighting',
    'steps': 20,
    'width': 512,
    'height': 512,
})
img_b64 = r.json()['images'][0]
display(Image(data=base64.b64decode(img_b64)))